In [1]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

c:\Users\ansul\OneDrive\Desktop\data science project\financial_rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
## Load chunks
with open(r"..\data\chunked\chunks_type1.json") as f:
    chunks = json.load(f)

In [3]:
## creating document object
documents = []

for c in chunks:
    doc = Document(
        page_content=c["text"],
        metadata={"company_name": c["company_name"], "section": c["section"], "chunk_id": c["chunk_id"]},
    )
    documents.append(doc)

In [4]:
## embedding model intialize
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en"
)

C:\Users\ansul\AppData\Local\Temp\ipykernel_15488\4207089419.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3718.01it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Creating index file for first time

In [ ]:
## building vector store
vectorstore = FAISS.from_documents(
    documents,
    embedding_model
)

vectorstore.save_local(r"..\data\vectorStore")

Saving index...


In [ ]:
vectorstore.save_local(r"..\data\embeddings")

## Use the local vector index

In [10]:
## loading the vector store
vectorstore = FAISS.load_local(
    r"..\data\embeddings",embeddings=embedding_model,allow_dangerous_deserialization=True
)

## Testing Retrival 

In [12]:
vectorstore.similarity_search("What is the revenue of Apple Inc. in 2020?", k=3)

[Document(id='52441168-ae80-49ff-b74a-a6c533f5ceec', metadata={'company_name': 'AAPL', 'section': 'management_discussion_and_analysis', 'chunk_id': 0}, page_content='s Discussion and Analysis of Financial Condition and Results of OperationsThe following discussion should be read in conjunction with the consolidated financial statements and accompanying notes included in Part II, Item 8 of this Form 10-K. This Item generally discusses 2023 and 2022 items and year-to-year comparisons between 2023 and 2022. Discussions of 2021 items and year-to-year comparisons between 2022 and 2021 are not included, and can be found in Managements Discussion and Analysis of Financial Condition and Results of Operations in Part II, Item 7 of the Companys Annual Report on Form 10-K for the fiscal year ended September 24, 2022. Fiscal PeriodThe Companys fiscal year is the 52- or 53-week period that ends on the last Saturday of September. An additional week is included in the first fiscal quarter every five 

In [15]:
vectorstore.similarity_search("What is the risks associated with netflix?", k=3)

[Document(id='bf9b15e7-5d53-442b-86ca-3e0dd3d6c669', metadata={'company_name': 'NFLX', 'section': 'risk_factors', 'chunk_id': 5}, page_content=" which compete directly with us or have investments in competing streaming content providers. In many instances, our agreements also include provisions by which the partner bills consumers directly for the Netflix service or otherwise offers services or products in connection with offering our service. If partners or other providers do a better job of connecting consumers with content they want to watch, for example through multi-service discovery interfaces, our service may be adversely impacted. We intend to continue to broaden our relationships with existing partners and to increase our capability to stream TV series and films to other platforms and partners over time. If we are not successful in maintaining existing and creating new relationships, or if we encounter technological, content licensing, regulatory, business or other impediments